In [15]:
import importlib
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "agentic_rag.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "agentic_rag.py").exists():
    raise FileNotFoundError("Open this notebook inside the agentic-rag-v2 project.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

import retrieval

importlib.reload(retrieval)
import agentic_rag

importlib.reload(agentic_rag)
answer_with_agent = agentic_rag.answer_with_agent
conversation = []


def ask_book(question: str):
    answer, sources, activity = answer_with_agent(question, history=conversation)
    conversation.extend(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    )

    print("Agent tool activity:")
    for step in activity:
        action_input = step.get("input", step.get("query", "(no input)"))
        result = step.get("result", f"{step.get('matches', 0)} matches")
        print(f"- {step.get('tool', 'tool')}: {action_input} -> {result}")

    print("\nAnswer:\n", answer)
    if sources:
        print("\nSources:")
        for source in sources:
            print(
                f"- [{source['id']}] {source['section']} "
                f"(pages {source['page_start']}-{source['page_end']})"
            )
    else:
        print("No book sources were used.")


assert "scope" in retrieval.retrieve_chunks.__code__.co_varnames
print("Agent and retrieval modules reloaded; scoped search is ready.")

Agent and retrieval modules reloaded; scoped search is ready.


In [17]:
ask_book("What is mind management?")

Agent tool activity:
- search_book: mind management definition -> Found 5 relevant passages

Answer:
 Mind management is the practice of optimizing your creative energy (your mind) rather than optimizing clock time. It’s about creating the internal conditions and mental states that let you have valuable thoughts and do the right work at the right moment, instead of treating every moment as equally productive [19a1c91f09bd-00049][19a1c91f09bd-00047].

Key points from the book:
- Mind management picks up where time-management systems leave off: time management optimizes time, mind management optimizes creative energy [19a1c91f09bd-00049].  
- It aims to make it obvious what to do at a given moment by managing your energy and mood (the author compares it to athletes warming up before peak performance) [19a1c91f09bd-00051].  
- A simple exercise: before working, ask “What work am I in the mood to do right now?” and “What do I need to do that fits that mood?” [19a1c91f09bd-00051].  
- The a